# Drug Offences in London: Spatial Regression Analysis

**Course:** 7CUSMSDA Spatial Data Analysis — Mini Project  
**Dependent variable:** Two-year cumulative drug offence rate per 1,000 residents (LSOA level, 2024–2025)  
**Independent variables:** IMD Score (2025), Population Density, % Young Males 15–29  
**Method chain:** OLS → Moran's I → Spatial Lag/Error → GWR

---
## Section 0: Imports & Paths

In [ ]:
# Standard
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
%matplotlib inline

# Spatial analysis (PySAL stack)
import esda
import spreg
from spreg import OLS, ML_Lag, ML_Error
from libpysal.weights import Queen

# GWR
from mgwr.sel_bw import Sel_BW
from mgwr.gwr import GWR

print('All libraries loaded.')

In [ ]:
# ── Project root (CourseWork/) ────────────────────────────────────────────────
CW = Path('..').resolve()

# ── Raw data ─────────────────────────────────────────────────────────────────
CRIME_DIR      = CW / 'data' / 'raw' / 'crime'
IMD_SHP        = CW / 'data' / 'raw' / 'IMD' / 'IMD_2025_LDN.shp'
POP_XLS        = CW / 'data' / 'raw' / 'nomis_2026_04_22_014758.xlsx'
DENSITY_XLS    = CW / 'data' / 'raw' / 'nomis_2026_04_22_011527.xlsx'
YOUNGMALE_CSV  = CW / 'data' / 'raw' / '1590321589052220.csv'
LSOA_SHP       = CW / 'data' / 'raw' / 'LSOA_london_2021' / 'LSOA_london_2021.shp'
GANGS_DIR      = CW / 'data' / 'raw' / 'London_Gangs'

# ── Outputs ───────────────────────────────────────────────────────────────────
PROCESSED_DIR  = CW / 'data' / 'processed'
FIGURES_DIR    = CW / 'figures'
OUTPUT_GPKG    = PROCESSED_DIR / 'london_drug_gdf.gpkg'

# ── Analysis constants ────────────────────────────────────────────────────────
# Target period: 2024-01 to 2025-12 (24 months)
TARGET_MONTHS  = [f'2024-{i:02d}' for i in range(1, 13)] + \
                 [f'2025-{i:02d}' for i in range(1, 13)]

# Variable names used in Sections 3–5
Y_NAME  = 'drug_rate'
X_NAMES = ['imd_score', 'pop_density', 'pct_young_male']
ANALYTICAL_COLS = [Y_NAME] + X_NAMES

# Verify key paths exist
for p in [CRIME_DIR, IMD_SHP, POP_XLS, DENSITY_XLS, YOUNGMALE_CSV, LSOA_SHP]:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'{status}  {p.name}')

---
## Section 1: Data Preparation

> **One-time setup.** Run this section once to generate `london_drug_gdf.gpkg`.  
> For all subsequent sessions, skip to **Section 2** and use the reload cell there.

This section loads all raw data sources, merges them onto the LSOA boundary, computes derived variables, and saves a single processed GeoPackage.

**Sources:**
| # | Data | Source | Role |
|---|------|--------|------|
| 1.1 | LSOA 2021 boundary | ONS / Week 10 practical | Master spatial frame |
| 1.2 | Crime records 2024–2025 | MPS open data (monthly CSVs) | Dependent variable (counts) |
| 1.3 | Total population | Census 2021 TS001 (Nomis) | Denominator for crime rate |
| 1.4 | Population density | Census 2021 TS006 (Nomis) | Independent variable X₂ |
| 1.5 | Young males 15–29 | Census 2021 RM200 (Nomis) | Independent variable X₃ |
| 1.6 | IMD 2025 | MHCLG / Week 3 practical | Independent variable X₁ |

### 1.1 LSOA 2021 Boundary (Master Frame)

All other data sources will be left-joined onto this GeoDataFrame.  
CRS must be EPSG:27700 (British National Grid, metres) for GWR with `spherical=False`.

In [ ]:
boundary = gpd.read_file(LSOA_SHP)

# Ensure EPSG:27700 (BNG)
if boundary.crs is None or boundary.crs.to_epsg() != 27700:
    boundary = boundary.to_crs(epsg=27700)
    print('Reprojected to EPSG:27700')

print(f'CRS          : {boundary.crs}')
print(f'LSOAs        : {len(boundary)}')
print(f'Columns      : {list(boundary.columns)}')
boundary.head(3)

### 1.2 Crime Data — Drug Offences (2024–2025)

MPS open crime records, 24 monthly CSVs.  
Filter to `Crime type == 'Drugs'` and to London LSOAs only, then aggregate to LSOA-level counts.  
LSOAs with **zero** drug offences are a valid observation — they receive `drug_count = 0` after the left join.

In [ ]:
london_codes = set(boundary['LSOA21CD'])

# Collect monthly CSV paths for target period only
crime_files = []
for month in TARGET_MONTHS:
    crime_files.extend((CRIME_DIR / month).glob('*.csv'))
crime_files = sorted(crime_files)
print(f'Monthly files found : {len(crime_files)}  (expected 24)')

# Load, filter to Drug offences + London LSOAs, concatenate
dfs = []
for f in crime_files:
    df = pd.read_csv(f, usecols=['LSOA code', 'Crime type'])
    df = df[(df['Crime type'] == 'Drugs') &
            (df['LSOA code'].isin(london_codes))]
    dfs.append(df)

crime_raw = pd.concat(dfs, ignore_index=True)
print(f'Drug offence records (London, 2024–2025): {len(crime_raw):,}')

# Aggregate to LSOA level
drug_counts = (crime_raw
               .groupby('LSOA code')
               .size()
               .reset_index(name='drug_count')
               .rename(columns={'LSOA code': 'LSOA21CD'}))
print(f'LSOAs with ≥1 drug offence : {len(drug_counts)}')

### 1.3 Total Population (Census 2021 TS001)

Used as the **denominator** when computing the drug offence rate per 1,000 population.  
Nomis LSOA labels are in the format `"E01000001 : Area Name"` — we extract the 9-character code.

In [ ]:
pop_raw = pd.read_excel(POP_XLS, skiprows=6, header=None,
                        names=['lsoa_label', 'total_pop'])

# Parse LSOA code (first 9 characters of label)
pop_raw['LSOA21CD'] = pop_raw['lsoa_label'].astype(str).str[:9]

# Filter to London and clean
pop = (pop_raw[pop_raw['LSOA21CD'].isin(london_codes)]
       [['LSOA21CD', 'total_pop']]
       .copy())
pop['total_pop'] = pd.to_numeric(pop['total_pop'], errors='coerce')

print(f'Population records (London): {len(pop)}')
print(pop.describe())

### 1.4 Population Density (Census 2021 TS006)

Already expressed as **usual residents per km²** — used directly as independent variable X₂.

In [ ]:
dens_raw = pd.read_excel(DENSITY_XLS, skiprows=6, header=None,
                         names=['lsoa_label', 'pop_density'])

dens_raw['LSOA21CD'] = dens_raw['lsoa_label'].astype(str).str[:9]

dens = (dens_raw[dens_raw['LSOA21CD'].isin(london_codes)]
        [['LSOA21CD', 'pop_density']]
        .copy())
dens['pop_density'] = pd.to_numeric(dens['pop_density'], errors='coerce')

print(f'Density records (London): {len(dens)}')
print(dens.describe())

### 1.5 Young Males Aged 15–29 (Census 2021 RM200, Male only)

This table records **male** residents by single year of age.  
We sum age columns 15–29 to get `young_male_count` per LSOA.  
The percentage (X₃) is computed in Section 1.7 once total population is available.

In [ ]:
ym_raw = pd.read_csv(YOUNGMALE_CSV, skiprows=8, low_memory=False)

# First column is the LSOA label
lsoa_col = ym_raw.columns[0]
ym_raw['LSOA21CD'] = ym_raw[lsoa_col].astype(str).str[:9]
ym_raw = ym_raw[ym_raw['LSOA21CD'].isin(london_codes)].copy()

# Sum single-year age columns 15–29
age_cols = [f'Aged {a} years' for a in range(15, 30)]
ym_raw['young_male_count'] = (
    ym_raw[age_cols]
    .apply(pd.to_numeric, errors='coerce')
    .sum(axis=1)
)

ym = ym_raw[['LSOA21CD', 'young_male_count']].copy()
print(f'Young male records (London): {len(ym)}')
print(ym['young_male_count'].describe())

### 1.6 IMD 2025 (MHCLG)

Already clipped to London (`LSOA21CD` field, 2021 boundaries).  
We use the composite `Index of M` score as independent variable X₁ — higher score = more deprived.

In [ ]:
imd_raw = gpd.read_file(IMD_SHP)

# Rename truncated field to a clean name
imd_raw = imd_raw.rename(columns={'Index of M': 'imd_score'})

# Keep only the join key and the score (drop geometry — plain DataFrame merge)
imd = pd.DataFrame(imd_raw[['LSOA21CD', 'imd_score']]).copy()
imd['imd_score'] = pd.to_numeric(imd['imd_score'], errors='coerce')

print(f'IMD 2025 records: {len(imd)}')
print(imd['imd_score'].describe())

### 1.7 Merge All Sources onto Boundary

Using left joins throughout — all 5,119 boundary LSOAs are retained.  
LSOAs with no drug records receive `drug_count = NaN` here, which is filled to `0` in the next step.

In [ ]:
gdf = (boundary
       .merge(drug_counts, on='LSOA21CD', how='left')
       .merge(pop,         on='LSOA21CD', how='left')
       .merge(dens,        on='LSOA21CD', how='left')
       .merge(ym,          on='LSOA21CD', how='left')
       .merge(imd,         on='LSOA21CD', how='left')
)

print(f'Merged GDF rows : {len(gdf)}  (should be {len(boundary)})')
print(f'Columns         : {list(gdf.columns)}')
print(f'\nNaN counts (pre-derived variables):')
print(gdf[['drug_count', 'total_pop', 'pop_density', 'young_male_count', 'imd_score']].isnull().sum())

### 1.8 Derived Variables

- **`drug_rate`** = drug offence count / total population × 1,000  
  LSOAs with zero offences get `drug_count = 0` (not NaN) before this step.  
- **`pct_young_male`** = young male count (15–29) / total population × 100  

Both use `np.where` to guard against division by zero (returns NaN for zero-population LSOAs).

In [ ]:
# Zero offences is a valid observation; NaN after left join means no records matched
gdf['drug_count'] = gdf['drug_count'].fillna(0)

# Drug offence rate per 1,000 population
gdf['drug_rate'] = np.where(
    gdf['total_pop'] > 0,
    (gdf['drug_count'] / gdf['total_pop']) * 1000,
    np.nan
)

# Percentage of young males (15–29) in total population
gdf['pct_young_male'] = np.where(
    gdf['total_pop'] > 0,
    (gdf['young_male_count'] / gdf['total_pop']) * 100,
    np.nan
)

print('Summary statistics for analytical variables:')
gdf[ANALYTICAL_COLS].describe().round(3)

### 1.9 Validate & Save

Run assertions before saving. Fix any issues here — do not proceed to analysis with unexpected NaN counts or row mismatches.

The `gdf_model` subset (NaN rows dropped, index reset) is what Sections 3–5 will use for regression.  
The full `gdf` (all LSOAs) is kept for choropleth mapping.

In [ ]:
# ── Validation ────────────────────────────────────────────────────────────────
assert len(gdf) == len(boundary), \
    f'Row count changed after merge: {len(gdf)} vs {len(boundary)}'

print('NaN audit (analytical columns):')
print(gdf[ANALYTICAL_COLS].isnull().sum())

# Spot-check City of London 001A
print('\nSpot-check E01000001:')
print(gdf.loc[gdf['LSOA21CD'] == 'E01000001', ANALYTICAL_COLS])

# ── Model subset (complete cases only) ───────────────────────────────────────
gdf_model = gdf.dropna(subset=ANALYTICAL_COLS).copy().reset_index(drop=True)
print(f'\nFull GDF   : {len(gdf)} LSOAs')
print(f'Model GDF  : {len(gdf_model)} LSOAs (dropped {len(gdf) - len(gdf_model)} NaN rows)')

# ── Save ──────────────────────────────────────────────────────────────────────
gdf.to_file(OUTPUT_GPKG, driver='GPKG')
print(f'\nSaved → {OUTPUT_GPKG}')

---
## Section 2: Exploratory Analysis

Before running any regression we inspect:
1. The distribution of each variable (skewness, outliers)
2. The spatial pattern of drug offences (to motivate spatial modelling)
3. The spatial patterns of each predictor
4. Bivariate relationships between Y and each X

> **Reload shortcut** — if Section 1 has already been run once, start here to skip re-processing raw data.

In [ ]:
# Fast reload — skip Section 1 if london_drug_gdf.gpkg already exists
gdf = gpd.read_file(OUTPUT_GPKG)
gdf_model = gdf.dropna(subset=ANALYTICAL_COLS).copy().reset_index(drop=True)
print(f'Loaded: {len(gdf_model)} LSOAs  |  CRS: {gdf.crs}')

In [ ]:
def choropleth(data, col, title, cmap='YlOrRd', scheme='quantiles',
               k=5, save=None, ax=None, figsize=(8, 8),
               missing_kwds=None, legend_kwds=None, **plot_kwargs):
    """Reusable choropleth. Pass ax= to embed in a subplot grid."""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=figsize)
    lk = {'loc': 'lower left', 'fontsize': 7, 'title': col}
    if legend_kwds:
        lk.update(legend_kwds)
    data.plot(
        column=col, cmap=cmap, scheme=scheme, k=k,
        legend=True, ax=ax,
        legend_kwds=lk,
        missing_kwds=missing_kwds,
        **plot_kwargs
    )
    ax.set_title(title, fontsize=11, pad=6)
    ax.set_axis_off()
    if standalone:
        plt.tight_layout()
        if save:
            plt.savefig(save, dpi=150, bbox_inches='tight')
        plt.show()

def plot_localr2_map(data, col, title, save=None, figsize=(9, 9)):
    """Continuous Local R^2 map with negatives clipped to 0 for display."""
    fig, ax = plt.subplots(figsize=figsize)
    vmax = np.nanpercentile(data[col], 98)
    vmax = max(float(vmax), 0.05)
    data.plot(
        column=col,
        cmap='PuBuGn',
        vmin=0,
        vmax=vmax,
        linewidth=0,
        legend=True,
        legend_kwds={'label': 'Local R^2 (negative raw values clipped to 0)', 'shrink': 0.65},
        ax=ax,
    )
    ax.set_title(title, fontsize=12, pad=8)
    ax.set_axis_off()
    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=150, bbox_inches='tight')
    plt.show()

def plot_gwr_coef_map(data, coef_col, sig_col, title, save=None,
                      overlay_gdfs=None, figsize=(9, 9)):
    """Continuous GWR coefficient map centered at 0, with significant areas outlined."""
    fig, ax = plt.subplots(figsize=figsize)
    coef_vals = data[coef_col].dropna().values
    vmax = np.nanpercentile(np.abs(coef_vals), 98)
    vmax = max(float(vmax), 0.05)
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    data.plot(ax=ax, color='#f2f2f2', linewidth=0)
    data.plot(
        column=coef_col,
        cmap='RdBu_r',
        norm=norm,
        linewidth=0,
        legend=True,
        legend_kwds={'label': 'Local coefficient (std. units)', 'shrink': 0.65},
        ax=ax,
    )

    sig_layer = data[data[sig_col]].copy()
    if len(sig_layer) > 0:
        sig_layer.boundary.plot(ax=ax, color='black', linewidth=0.25, alpha=0.8)

    if overlay_gdfs:
        for ggang in overlay_gdfs:
            ggang.boundary.plot(ax=ax, color='#4d4d4d', linewidth=0.45, alpha=0.35)

    ax.set_title(title, fontsize=12, pad=8)
    ax.set_axis_off()
    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=150, bbox_inches='tight')
    plt.show()

### 2.1 Variable Distributions

Key things to note:
- `drug_rate` is heavily right-skewed (max = 482, mean = 10). High-rate outliers are small-population LSOAs with a cluster of incidents. This will likely trigger **Koenker-Bassett heteroskedasticity** in OLS.
- `pop_density` is also extremely right-skewed (City of London / Soho effect).
- `imd_score` and `pct_young_male` are more symmetric.

In [ ]:
titles = {
    'drug_rate':      'Two-Year Cumulative Drug Offence Rate\n(per 1,000 residents)',
    'imd_score':      'IMD Score 2025\n(higher = more deprived)',
    'pop_density':    'Population Density\n(persons per km²)',
    'pct_young_male': '% Young Males 15–29\n(of total population)',
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, (col, title) in enumerate(titles.items()):
    ax = axes[i]
    gdf_model[col].hist(bins=50, ax=ax, color='steelblue', edgecolor='white', linewidth=0.3)
    ax.axvline(gdf_model[col].mean(),   color='red',    linestyle='--', linewidth=1.2, label=f'mean={gdf_model[col].mean():.1f}')
    ax.axvline(gdf_model[col].median(), color='orange', linestyle='--', linewidth=1.2, label=f'median={gdf_model[col].median():.1f}')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('')
    ax.legend(fontsize=8)

fig.suptitle('Distribution of Analytical Variables (n = 5,119 LSOAs)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Spatial Distribution of Drug Offence Rate

Quantile classification (5 classes) is used because the distribution is strongly right-skewed — equal-interval classes would compress nearly all LSOAs into the lowest band.

This map shows the **observed two-year cumulative rate**, not model residuals.  
Visible clustering would suggest spatial autocorrelation — which we will formally test in Section 4.

In [ ]:
choropleth(
    gdf_model, 'drug_rate',
    'Two-Year Cumulative Drug Offence Rate per 1,000 Residents\n(London LSOAs, Jan 2024-Dec 2025, Quantiles)',
    cmap='YlOrRd', scheme='quantiles', k=5,
    save=FIGURES_DIR / 'map_drug_rate.png'
)

### 2.3 Spatial Distribution of Independent Variables

Visual inspection before regression: do the predictor maps show spatial patterns that align with the drug rate map?

In [ ]:
x_config = [
    ('imd_score',      'IMD Score 2025\n(Quantiles)', 'OrRd'),
    ('pop_density',    'Population Density (persons/km²)\n(Quantiles)', 'Blues'),
    ('pct_young_male', '% Young Males 15–29\n(Quantiles)', 'Purples'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, (col, title, cmap) in zip(axes, x_config):
    choropleth(gdf_model, col, title, cmap=cmap, scheme='quantiles', k=5, ax=ax)

fig.suptitle('Independent Variables — Spatial Distribution', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'map_predictors.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4 Bivariate Relationships (Y vs Each X)

Scatter plots with OLS trend lines to assess linearity before formal regression.  
Note: extreme `drug_rate` outliers are visible — their influence on the regression line will be assessed via the OLS diagnostic statistics in Section 3.

In [ ]:
x_labels = {
    'imd_score':      'IMD Score 2025',
    'pop_density':    'Population Density (persons/km²)',
    'pct_young_male': '% Young Males 15–29',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, xvar in zip(axes, X_NAMES):
    x = gdf_model[xvar].values
    y = gdf_model[Y_NAME].values

    ax.scatter(x, y, alpha=0.15, s=4, color='steelblue')

    # OLS trend line
    m, b = np.polyfit(x, y, 1)
    xline = np.linspace(x.min(), x.max(), 200)
    slope_txt = f'{m:.2e}' if abs(m) < 0.01 else f'{m:.3f}'
    ax.plot(xline, m * xline + b, color='red', linewidth=1.5,
            label=f'slope = {slope_txt}')

    ax.set_xlabel(x_labels[xvar], fontsize=9)
    ax.set_ylabel('Two-Year Drug Offence Rate (per 1,000)' if xvar == 'imd_score' else '', fontsize=9)
    ax.set_title(f'{Y_NAME} vs {xvar}', fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle('Bivariate Relationships: Two-Year Drug Rate vs Predictors', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'scatter_bivariate.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3: OLS Regression

OLS establishes the **baseline (non-spatial) relationship** between the socio-economic predictors and drug offence rates.

Key outputs:
- Coefficient estimates and significance
- R² (overall fit)
- **Robust LM tests** → tell us Spatial Lag vs Spatial Error for Section 5
- Jarque-Bera / Koenker-Bassett → residual diagnostics

The spatial weights matrix `w` built here is reused in Sections 4 and 5.

### 3.1 Spatial Weights Matrix (Queen Contiguity)

Queen contiguity: two LSOAs are neighbours if they share any boundary point (edge or corner).  
`w.transform = 'r'` row-standardises — required before Moran's I and all spatial regression models.

In [ ]:
w = Queen.from_dataframe(gdf_model)
w.transform = 'r'
print(f'Weights matrix  : Queen contiguity, row-standardised')
print(f'n               : {w.n} LSOAs')
print(f'Mean neighbours : {w.mean_neighbors:.2f}')
print(f'Islands (n=0)   : {len(w.islands)}')

### 3.2 Prepare Regression Arrays

`y` and `X` must be numpy arrays. Variable names from Section 0 (`Y_NAME`, `X_NAMES`) are used throughout.

In [ ]:
y = gdf_model[Y_NAME].values.reshape(-1, 1)
X = gdf_model[X_NAMES].values
print(f'y shape : {y.shape}   ({Y_NAME})')
print(f'X shape : {X.shape}   {X_NAMES}')

### 3.3 OLS Regression

`w=w, spat_diag=True` is required so that spatial diagnostics (LM tests) appear in the summary and are accessible as attributes.  
spreg adds a constant automatically — do **not** add it to `X` manually.

In [ ]:
ols = OLS(y, X,
          w=w,
          name_y=Y_NAME,
          name_x=X_NAMES,
          name_ds='london_drug_lsoa',
          spat_diag=True)
print(ols.summary)

### 3.4 Multicollinearity Check (VIF)

VIF (Variance Inflation Factor): VIF < 5 = no issue, 5–10 = moderate, > 10 = problematic.

In [ ]:
def compute_vif(X_arr, names):
    """VIF = 1 / (1 - R²) from regressing each X on all others."""
    n, k = X_arr.shape
    rows = []
    for i in range(k):
        y_i   = X_arr[:, i]
        X_oth = np.delete(X_arr, i, axis=1)
        X_c   = np.column_stack([np.ones(n), X_oth])
        coef, _, _, _ = np.linalg.lstsq(X_c, y_i, rcond=None)
        ss_res = np.sum((y_i - X_c @ coef) ** 2)
        ss_tot = np.sum((y_i - y_i.mean()) ** 2)
        r2  = 1 - ss_res / ss_tot
        vif = 1 / (1 - r2) if r2 < 1 else np.inf
        rows.append({'Variable': names[i], 'VIF': round(vif, 3)})
    return pd.DataFrame(rows)

print(compute_vif(X, X_NAMES).to_string(index=False))

### 3.5 Diagnostic Summary

Extract key statistics from the OLS object for reference.  
When **both** raw LM tests are significant (common with large n), use the **Robust LM** tests to choose between Spatial Lag and Spatial Error — pick the model with the higher Robust LM statistic.

In [ ]:
print('─' * 50)
print(f'  R²             : {ols.r2:.4f}')
print(f'  Adjusted R²    : {ols.ar2:.4f}')
print('─' * 50)
print('  Residual diagnostics')
print(f'  Jarque-Bera    : stat={ols.jarque_bera["jb"]:.2f}   p={ols.jarque_bera["pvalue"]:.4f}')
print(f'  Koenker-Bassett: stat={ols.koenker_bassett["kb"]:.2f}   p={ols.koenker_bassett["pvalue"]:.4f}')
print('─' * 50)
print('  Spatial dependence (raw LM)')
print(f'  LM Lag         : stat={ols.lm_lag[0]:.2f}   p={ols.lm_lag[1]:.4e}')
print(f'  LM Error       : stat={ols.lm_error[0]:.2f}   p={ols.lm_error[1]:.4e}')
print('  Robust LM (tiebreaker when both raw LM are significant)')
print(f'  Robust LM Lag  : stat={ols.rlm_lag[0]:.2f}   p={ols.rlm_lag[1]:.4f}')
print(f'  Robust LM Error: stat={ols.rlm_error[0]:.2f}   p={ols.rlm_error[1]:.4f}')
print('─' * 50)

# Decision: use Robust LM when both raw LM tests are significant
lm_lag_sig  = ols.lm_lag[1]   < 0.05
lm_err_sig  = ols.lm_error[1] < 0.05
rlml_p = ols.rlm_lag[1]
rlme_p = ols.rlm_error[1]

if lm_lag_sig and lm_err_sig:
    winner = 'Spatial LAG' if ols.rlm_lag[0] > ols.rlm_error[0] else 'Spatial ERROR'
    print(f'  Both raw LM significant → compare Robust LM → use {winner} model')
elif lm_lag_sig:
    print('  LM Lag only → use Spatial LAG model')
elif lm_err_sig:
    print('  LM Error only → use Spatial ERROR model')
else:
    print('  Neither LM significant → OLS may be sufficient')

### 3.6 OLS Residual Map

Spatial pattern in residuals shows where the model under- or over-predicts.  
Visible geographic structure here is consistent with the LM diagnostics above and motivates the spatial models fitted in Section 5.

In [ ]:
gdf_model['ols_residuals'] = ols.u.flatten()

fig, ax = plt.subplots(figsize=(9, 9))
vlim = 30   # clip to ±30 so extreme outliers don't collapse the palette
gdf_model.plot(
    column='ols_residuals',
    cmap='RdBu_r',
    vmin=-vlim, vmax=vlim,
    legend=True,
    legend_kwds={'label': 'Residual (clipped ±30)', 'shrink': 0.6},
    ax=ax
)
ax.set_title('OLS Residuals\nRed = under-predicted, Blue = over-predicted', fontsize=11)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ols_residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Residual range: {gdf_model["ols_residuals"].min():.1f} to {gdf_model["ols_residuals"].max():.1f}')

---
## Section 4: Spatial Pattern of the Observed Drug Offence Rate

This section answers the first two proposal questions directly by testing the **observed** two-year cumulative drug offence rate for clustering before turning to model-based spatial regression.

- **Global Moran's I** — overall clustering in the observed rate
- **LISA (Local Moran's I)** — identifies specific hotspots, coldspots, and spatial outliers in the observed rate

These results describe the dependent variable itself. The need for spatial regression is established separately in Section 3 via the LM diagnostics and carried forward in Section 5.

### 4.1 Global Moran's I on the Observed Drug Offence Rate

In [ ]:
mi_rate = esda.Moran(gdf_model[Y_NAME].values, w, two_tailed=False)
print(f"Global Moran's I = {mi_rate.I:.4f}")
print(f"Expected (random): {mi_rate.EI:.4f}")
print(f"p-value (normal)  = {mi_rate.p_norm:.4f}")
print()
if mi_rate.p_norm < 0.05:
    print("→ Significant positive spatial autocorrelation in the observed drug offence rate.")
    print("  Similar rates cluster geographically rather than being randomly distributed.")
else:
    print("→ No significant spatial autocorrelation detected in the observed rate.")

### 4.2 LISA Cluster Map of the Observed Rate

LISA decomposes the global Moran's I into local statistics, identifying where clustering occurs in the **observed** drug offence rate:
- **HH** (High-High): high drug rate surrounded by high-rate neighbours — hotspots
- **LL** (Low-Low): low drug rate surrounded by low-rate neighbours — coldspots
- **HL** (High-Low): high-rate LSOA surrounded by low-rate neighbours — spatial outlier
- **LH** (Low-High): low-rate LSOA surrounded by high-rate neighbours — spatial outlier

In [ ]:
lisa_rate = esda.Moran_Local(gdf_model[Y_NAME].values, w, permutations=999)
gdf_model['rate_lisa_q'] = lisa_rate.q       # quadrant: 1=HH, 2=LH, 3=LL, 4=HL
gdf_model['rate_lisa_p'] = lisa_rate.p_sim   # simulated p-value

sig = gdf_model['rate_lisa_p'] < 0.05
gdf_model['rate_lisa_cluster'] = 'Not significant'
gdf_model.loc[sig & (gdf_model['rate_lisa_q'] == 1), 'rate_lisa_cluster'] = 'HH'
gdf_model.loc[sig & (gdf_model['rate_lisa_q'] == 2), 'rate_lisa_cluster'] = 'LH'
gdf_model.loc[sig & (gdf_model['rate_lisa_q'] == 3), 'rate_lisa_cluster'] = 'LL'
gdf_model.loc[sig & (gdf_model['rate_lisa_q'] == 4), 'rate_lisa_cluster'] = 'HL'

print('LISA cluster counts (observed rate):')
print(gdf_model['rate_lisa_cluster'].value_counts())

In [ ]:
cluster_colours = {
    'HH': '#d7191c',
    'LH': '#abd9e9',
    'LL': '#2c7bb6',
    'HL': '#fdae61',
    'Not significant': '#eeeeee',
}
order = ['HH', 'HL', 'LH', 'LL', 'Not significant']

fig, ax = plt.subplots(figsize=(9, 9))
for label in order:
    subset = gdf_model[gdf_model['rate_lisa_cluster'] == label]
    subset.plot(ax=ax, color=cluster_colours[label], label=label, linewidth=0.1)

ax.set_title("LISA Cluster Map — Observed Drug Offence Rate\n(Local Moran's I, p < 0.05)", fontsize=12)
ax.set_axis_off()
ax.legend(title='Cluster', loc='lower left', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'lisa_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5: Spatial Error Model & GWR

### 5.1 Spatial Error Model

The Robust LM Error test (stat=31.15) outperformed Robust LM Lag (17.48), indicating that spatial dependence is best captured as **unmeasured spatially correlated factors** in the error term rather than a direct spatial spillover in the dependent variable.

The Spatial Error model corrects for this by including a spatially lagged error term (λ·Wε).

In [ ]:
serr = ML_Error(y, X, w,
                name_y=Y_NAME,
                name_x=X_NAMES,
                name_w='w_queen',
                name_ds='london_drug_lsoa')
print(serr.summary)

### 5.2 Compare OLS vs Spatial Models

Fit the screened-in Spatial Lag model alongside the preferred Spatial Error model so the final choice is transparent.

In [ ]:
slag = ML_Lag(y, X, w,
              name_y=Y_NAME,
              name_x=X_NAMES,
              name_w='w_queen',
              name_ds='london_drug_lsoa')

print(f"{'Model':<20} {'Log-Likelihood':>16} {'AIC':>12}")
print('-' * 50)
print(f"{'OLS':<20} {ols.logll:>16.3f} {ols.aic:>12.3f}")
print(f"{'Spatial Lag':<20} {slag.logll:>16.3f} {slag.aic:>12.3f}")
print(f"{'Spatial Error':<20} {serr.logll:>16.3f} {serr.aic:>12.3f}")
print()
print(f"Preferred final spatial model: {'Spatial Error' if serr.aic < slag.aic else 'Spatial Lag'}")
print(f"AIC improvement over OLS (SEM): {ols.aic - serr.aic:.1f}")
print(f"AIC improvement over OLS (SLM): {ols.aic - slag.aic:.1f}")

### 5.3 GWR — Exploratory Spatial Non-stationarity Check

GWR is used here as an **exploratory follow-up** after the global spatial models.  
It helps assess whether relationships may vary across London, but the local maps should only be interpreted where coefficients remain significant after multiple-testing adjustment.

**Setup:**
- Projected CRS (EPSG:27700, metres) → `spherical=False`  
- Adaptive **bisquare** kernel with bandwidth selected by **AICc**  
- Both X and y are standardised (mean=0, std=1) before fitting for numerical stability  
- Local coefficient maps are filtered using the adjusted GWR critical t-value before interpretation

In [ ]:
# Centroids (EPSG:27700, metres)
u_coords = gdf_model.geometry.centroid.x.values
v_coords = gdf_model.geometry.centroid.y.values
coords = list(zip(u_coords, v_coords))

# Standardise (required for GWR numerical stability)
X_s = (X - X.mean(axis=0)) / X.std(axis=0)
y_s = (y - y.mean()) / y.std()

print(f"Coordinates range: x={u_coords.min():.0f}–{u_coords.max():.0f}, "
      f"y={v_coords.min():.0f}–{v_coords.max():.0f}")
print("Standardised X stats (should be ~mean=0, std=1):")
print(f"  means : {X_s.mean(axis=0).round(6)}")
print(f"  stds  : {X_s.std(axis=0).round(3)}")

#### Bandwidth Selection

`Sel_BW` finds the optimal number of nearest neighbours by minimising **AICc**.  
This may take a few minutes for n=5,119.

In [ ]:
bw_selector = Sel_BW(coords, y_s, X_s, fixed=False, spherical=False, n_jobs=1)
bw_selector.search(bw_min=2, criterion='AICc')
opt_bw = bw_selector.bw[0]
print(f"Optimal adaptive bandwidth (AICc): {opt_bw} nearest neighbours")

In [ ]:
gwr_model = GWR(coords, y_s, X_s, opt_bw, n_jobs=1)
gwr_results = gwr_model.fit()

filtered_t = gwr_results.filter_tvals()
sig_ratio = (np.abs(filtered_t[:, 1:]) > 0).mean()

print(f"GWR R² (global)      : {gwr_results.R2:.4f}")
print(f"GWR adjusted R²      : {gwr_results.adj_R2:.4f}")
print(f"Effective parameters : {gwr_results.ENP:.2f}")
print(f"Local R² (raw)       : min={np.min(gwr_results.localR2):.3f}, mean={np.mean(gwr_results.localR2):.3f}, max={np.max(gwr_results.localR2):.3f}")
print(f"Adjusted critical |t|: {gwr_results.critical_tval():.3f}")
print(f"Share of significant local X coefficients: {sig_ratio:.3%}")
print()
print(f"{'Model':<20} {'AIC':>10} {'AICc':>10}")
print('-' * 44)
print(f"{'OLS':<20} {ols.aic:>10.1f} {'n/a':>10}")
print(f"{'Spatial Error':<20} {serr.aic:>10.1f} {'n/a':>10}")
print(f"{'GWR':<20} {gwr_results.aic:>10.1f} {gwr_results.aicc:>10.1f}")
print(f"\nAIC improvement over Spatial Error: {serr.aic - gwr_results.aic:.1f}")
print('Interpretation note: lower AIC supports GWR as a better descriptive fit, but local coefficient maps should only be interpreted where coefficients remain significant after adjustment.')

In [ ]:
# Store raw coefficients plus multiple-testing-adjusted significance masks
labels = ['Intercept'] + X_NAMES   # index 0 = Intercept, 1,2,3 = X vars
critical_t = gwr_results.critical_tval()

for i, name in enumerate(labels):
    gdf_model[f'coef_{name}'] = gwr_results.params[:, i]
    gdf_model[f'tval_{name}'] = gwr_results.tvalues[:, i]
    gdf_model[f'tvalf_{name}'] = filtered_t[:, i]
    gdf_model[f'sig_{name}'] = np.abs(filtered_t[:, i]) > 0
    gdf_model[f'coef_{name}_sig'] = np.where(
        gdf_model[f'sig_{name}'],
        gdf_model[f'coef_{name}'],
        np.nan
    )

gdf_model['localR2'] = gwr_results.localR2
gdf_model['localR2_plot'] = gdf_model['localR2'].clip(lower=0)

summary_rows = []
for name in X_NAMES:
    summary_rows.append({
        'Variable': name,
        'Coef mean': round(gdf_model[f'coef_{name}'].mean(), 4),
        'Coef min': round(gdf_model[f'coef_{name}'].min(), 4),
        'Coef max': round(gdf_model[f'coef_{name}'].max(), 4),
        'Significant share': round(gdf_model[f'sig_{name}'].mean(), 4),
    })

print(f'Adjusted critical |t| for mapped coefficients: {critical_t:.3f}')
print('GWR coefficient summary (standardised units):')
print(pd.DataFrame(summary_rows).to_string(index=False))
print(f"Raw local R² range: {gdf_model['localR2'].min():.3f} to {gdf_model['localR2'].max():.3f}")

### 5.4 GWR Maps — Continuous Coefficient Surfaces with Significant Areas Marked

In [ ]:
# Local R^2 map
plot_localr2_map(
    gdf_model,
    'localR2_plot',
    'GWR - Local R^2',
    save=FIGURES_DIR / 'gwr_localR2.png'
)

In [ ]:
# Continuous coefficient maps, with adjusted-significance areas outlined
x_titles = {
    'imd_score':      'GWR Coefficient - IMD Score 2025\n(black outlines = adjusted local significance)',
    'pop_density':    'GWR Coefficient - Population Density\n(black outlines = adjusted local significance)',
    'pct_young_male': 'GWR Coefficient - % Young Males 15-29\n(black outlines = adjusted local significance)',
}

for xvar in X_NAMES:
    plot_gwr_coef_map(
        gdf_model,
        f'coef_{xvar}',
        f'sig_{xvar}',
        x_titles[xvar],
        save=FIGURES_DIR / f'gwr_coef_{xvar}.png'
    )

### 5.5 Gang Territory Overlay on the Young-Male Coefficient Map

The gang territory layer is shown only as **visual context** for the `% young male` GWR coefficient surface.  
Black outlines still mark adjusted local significance, while the gang boundaries are shown as muted dark-grey context only.  
This is **not** a formal overlap test or causal claim, so any apparent coincidence should be interpreted cautiously.

In [ ]:
import glob

# Load all gang territory shapefiles and reproject to match gdf_model CRS
gang_files = sorted(glob.glob(str(GANGS_DIR / '*.shp')))
print(f"Gang shapefiles found: {len(gang_files)}")
for f in gang_files:
    print(f"  {Path(f).name}")

gang_gdfs = [gpd.read_file(f).to_crs(gdf_model.crs) for f in gang_files]

In [ ]:
plot_gwr_coef_map(
    gdf_model,
    'coef_pct_young_male',
    'sig_pct_young_male',
    'GWR Coefficient: % Young Males 15-29\n(black outlines = adjusted local significance; dark-grey outlines = gang context)',
    save=FIGURES_DIR / 'gwr_coef_youngmale_gangs.png',
    overlay_gdfs=gang_gdfs
)